# TRELLIS.2 — Drive Batch Image → 3D

## Use
1. Put images in **`MyDrive/AI_Projects/TRELLIS.2/TRELLIS_INPUT`**.
2. In Colab select a **GPU runtime**.
3. Click **Runtime → Run all**.

That is it. The notebook installs/loads TRELLIS once, processes every PNG/JPG/JPEG/WEBP in the input folder, saves each model as a `.glb`, and stops when the input folder is empty.

### Drive folders
- Input: `MyDrive/AI_Projects/TRELLIS.2/TRELLIS_INPUT`
- Output: `MyDrive/AI_Projects/TRELLIS.2/TRELLIS_OUTPUT`
- Finished inputs: `MyDrive/AI_Projects/TRELLIS.2/TRELLIS_DONE`
- Failed inputs: `MyDrive/AI_Projects/TRELLIS.2/TRELLIS_FAILED`
- Build cache: `MyDrive/AI_Projects/TRELLIS.2/AI3D_Engine_Cache`

> If `HF_TOKEN` is already stored in Colab Secrets, there are no prompts. Otherwise Colab asks for the Hugging Face read token once.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass, os, shutil, subprocess

# Mount Drive and prepare the exact folders used by this workflow.
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/AI_Projects/TRELLIS.2')
INPUT_DIR = ROOT / 'TRELLIS_INPUT'
OUTPUT_DIR = ROOT / 'TRELLIS_OUTPUT'
DONE_DIR = ROOT / 'TRELLIS_DONE'
FAILED_DIR = ROOT / 'TRELLIS_FAILED'
CACHE_DIR = ROOT / 'AI3D_Engine_Cache'
for folder in (INPUT_DIR, OUTPUT_DIR, DONE_DIR, FAILED_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('Input :', INPUT_DIR)
print('Output:', OUTPUT_DIR)

# Pull the maintained TRELLIS installer + batch runner.
REPO = Path('/content/My-works')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/Logan17de/My-works.git', str(REPO)
], check=True)
TOOLS_3D = REPO / 'ai-3d-animation-engines' / '3d-engine'

# Persist native build/download cache in your TRELLIS.2 Drive folder.
os.environ['ENGINE_CACHE_ROOT'] = str(CACHE_DIR)
subprocess.run(['bash', str(TOOLS_3D / 'install_3d.sh')], check=True)

# Hugging Face access for TRELLIS dependencies.
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')

env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN
env['HF_HOME'] = '/content/huggingface'
env['HF_XET_HIGH_PERFORMANCE'] = '1'
env['PYTHONUNBUFFERED'] = '1'

subprocess.run([
    '/opt/conda/bin/conda', 'run', '--no-capture-output', '-n', 'trellis2',
    'python', str(TOOLS_3D / 'prepare_hf_models.py')
], cwd='/content/TRELLIS.2', env=env, check=True)

# Process the complete Drive input folder. Model loads once inside this runner.
subprocess.run([
    '/opt/conda/bin/conda', 'run', '--no-capture-output', '-n', 'trellis2',
    'python', str(TOOLS_3D / 'batch_drive_trellis2.py')
], cwd='/content/TRELLIS.2', env=env, check=True)
